In [3]:
"""
M-Pesa Safaricom Statement extraction tool
"""

import re, csv, pdfplumber
import pandas as pd

SKIP_LINES = [
    "M-PESA STATEMENT", "Customer Name:", "Mobile Number:", "Email Address:",
    "Statement Period:", "Request Date:", "SUMMARY", "DETAILED STATEMENT",
    "TRANSACTION TYPE", "PAID IN", "PAID OUT", "SEND MONEY", "RECEIVED MONEY",
    "AGENT DEPOSIT", "AGENT WITHDRAWAL", "LIPA NA M-PESA", "OTHERS", "TOTAL",
    "Receipt No.", "Completion Time", "Details", "Transaction Status",
    "Withdrawn", "Balance", "Completed", "www.safaricom", "Approved",
    "Safaricom", "P.O.BOX", "Simple", "FOR YOU", "Page",
    "Statement Verification Code", "To verify the validity",
    "prompts to enter the code", "For self-help dial", "Web:", "Twitter:",
    "Facebook:", "Terms and conditions apply", "Disclaimer", "Data Protection Act",
    "for which it was provided"
]

HEADERS = ["TXN_DATE", "VALUE_DATE", "DESCRIPTION", "MONEY_OUT_OR_IN", "BALANCE", "TXN_TYPE"]

PATTERN = re.compile(
    r"^([A-Z0-9]+)\s+"
    r"(\d{4}-\d{2}-\d{2})\s+"
    r"\d{2}:\d{2}:\d{2}\s+"
    r"(.+?)\s+"
    r"Completed\s+"
    r"(-?[\d,]+\.\d{2})\s+"
    r"(-?[\d,]+\.\d{2})$"
)


def format_date(raw):
    year, month, day = raw.split("-")
    return f"{int(day)}/{int(month)}/{year}"


def clean_amount(value):
    if not value or value.strip() in ("", "-"):
        return 0.0
    return float(value.replace(",", "").strip())


def clean_description(desc):
    return re.sub(r"\s*,\s*", " ", desc).strip()


def read_pdf(pdf_path, password):
    lines = []
    with pdfplumber.open(pdf_path, password=password) as pdf:
        for page in pdf.pages:
            lines.extend((page.extract_text() or "").splitlines())
    return lines


def get_metadata(lines):
    def find(label):
        for line in lines:
            if label in line:
                return line.split(label)[-1].strip()
        return ""
    return {
        "account_owner": find("Customer Name:"),
        "account_no":    find("Mobile Number:"),
        "currency":      "KES",
    }


def get_transactions(lines):
    transactions = []
    for line in lines:
        line  = line.strip()
        match = PATTERN.match(line)
        if match:
            receipt, date, desc, amount, balance = match.groups()

            amount_val = clean_amount(amount)
            txn_type   = "CR" if amount_val > 0 else "DR"

            transactions.append({
                "TXN_DATE":        format_date(date),
                "VALUE_DATE":      format_date(date),
                "DESCRIPTION":     receipt + " " + clean_description(desc),
                "MONEY_OUT_OR_IN": abs(amount_val),
                "BALANCE":         clean_amount(balance),
                "TXN_TYPE":        txn_type,
            })
        elif transactions and line and not any(s in line for s in SKIP_LINES):
            if not line.startswith("http") and not PATTERN.match(line):
                transactions[-1]["DESCRIPTION"] = clean_description(
                    transactions[-1]["DESCRIPTION"] + " " + line
                )

    if not transactions:
        raise ValueError("No transactions found — check the PDF format.")

    df = pd.DataFrame(transactions)
    df["MONEY_OUT_OR_IN"] = pd.to_numeric(df["MONEY_OUT_OR_IN"], errors="coerce")
    df["BALANCE"]         = pd.to_numeric(df["BALANCE"],         errors="coerce")
    return df


def write_csv(df, info, output_path):
    metadata = [
        f"StatementStartDate:{df['TXN_DATE'].iloc[-1]}",
        f"StatementEndDate:{df['TXN_DATE'].iloc[0]}",
        f"AccountNo:{info['account_no']}",
        f"Currency:{info['currency']}",
        f"AccountOwner:{info['account_owner']}",
    ]
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([""] + HEADERS)
        for i, item in enumerate(metadata):
            writer.writerow([i, "", "", item, "", "", ""])
        writer.writerow([len(metadata)] + HEADERS)
        for i, row in df.iterrows():
            writer.writerow([i + len(metadata) + 1] + row.tolist())


def run(pdf_path, output_path, password):
    lines = read_pdf(pdf_path, password)
    df    = get_transactions(lines)
    info  = get_metadata(lines)
    write_csv(df, info, output_path)
    print(f"Done — {len(df)} transactions saved to {output_path}")


run("Mpesa_statement.pdf", "Mpesa.csv", password="543343")

Done — 6013 transactions saved to Mpesa.csv


In [1]:
import pdfplumber

with pdfplumber.open("Mpesa_statement.pdf", password="543343") as pdf:
    for page in pdf.pages:
        for line in (page.extract_text() or "").splitlines():
            if "Completed" in line:
                print(repr(line))
                break
        break

'UDR8W25BP5 2026-04-27 13:11:58 Funds received from - Completed 50.00 50.00'
